# 4.2 基础 GEMM实现

## 概述

把一行一列的乘加组织成矩阵块，是从 Vector 走向 Cube 的关键。Cube 提供矩阵乘专用计算能力，本章会解释矩阵计算涉及到的搬入搬出流及 Cube 计算单元，并实现一个基础版本的 GEMM 来帮助大家理解。

## 学习目标

- 了解 Cube 单元中的 L1 与 L0C。
- 根据 M/N/K 计算分核和 K 循环切分。
- 掌握 T.gemm 的各项参数的意义。
- 构造用例并验证基础版实现的正确性。

[本章导航](README.md) · [上一节](04.01_intro.ipynb) · [下一节](04.03_pipeline.ipynb)

<p><small>执行前完成第1.4节环境准备，按本章README的统一入口运行。每页使用新内核；NPU程序由独立子进程执行，notebook内核不初始化设备。</small></p>

## 1. Cube 单元中的内存与流水

第3章的 Vector 算子把输入搬到 UB 后进行向量计算。矩阵乘使用另一条数据通路：A、B 在 GM 中保存；输入块先搬入 L1，再进入 Cube 的操作数内存 L0A、L0B；Cube 执行矩阵乘加，结果累加在 L0C 中，最后写回 GM。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">流水</th>
<th style="text-align: left; vertical-align: top;">本例承担的工作</th>
<th style="text-align: left; vertical-align: top;">数据位置</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">MTE2</td>
<td style="text-align: left; vertical-align: top;">从 GM 搬入输入块</td>
<td style="text-align: left; vertical-align: top;">GM → L1</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">MTE1</td>
<td style="text-align: left; vertical-align: top;">为矩阵乘准备操作数</td>
<td style="text-align: left; vertical-align: top;">L1 → L0A / L0B</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">Cube（矩阵计算流水）</td>
<td style="text-align: left; vertical-align: top;">矩阵乘加</td>
<td style="text-align: left; vertical-align: top;">L0A、L0B → L0C</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">FIX（Fixpipe）</td>
<td style="text-align: left; vertical-align: top;">搬出累加结果</td>
<td style="text-align: left; vertical-align: top;">L0C → GM</td>
</tr>
</tbody>
</table>

流水是执行不同类型指令的硬件通路，L1、L0A、L0B、L0C 则是存放数据的内存。不同流水可以具备并行工作的条件，但仍要满足数据依赖：输入准备好后才能计算，全部 K 块累加完成后才能写回。本节先完成这些步骤，4.3 再讨论核内多个输出块的流水排布。

![Cube 的内存与流水](images/04.02_cube_path.svg)

A、B、C 是 GM 中的三个矩阵；a、b 是 L1 中的两个输入块；c 是 L0C 中的输出块累加器。箭头上的名称表示执行搬运或计算的流水。

### 本章使用的内存分配写法

`T.alloc_l1` 为 a、b 分别分配 L1 内存，`T.alloc_l0c` 为 c 分配 fp32 累加内存。本例调用的 L1 版本 `T.gemm` 由后端进行 L1 到 L0A/L0B 的搬运和 Cube 计算，因此代码没有手写 L0A、L0B 的分配。L0C 保存的是不断更新的部分和，是 Cube 计算的结果。

```python
a = T.alloc_l1((BM, K), "bfloat16")
b = T.alloc_l1((BN, K), "bfloat16")
c = T.alloc_l0c((BM, BN), "float32")
```

### 对照通用 API 写法

也可以使用 `T.alloc_shared`、`T.alloc_fragment` 表达相同的内存分配，但需要显式指定内存位置，并为 L0C 配置布局。下面两组代码中的分配语句均放在 `T.Kernel` 内，二选一即可。

对应的通用 API 写法（布局函数在 Python 文件开头导入）：

```python
from tilelang.layout import make_ascend_l0c_layout

# 以下语句放在 T.Kernel 内
a = T.alloc_shared((BM, K), "bfloat16", scope="shared.l1")
b = T.alloc_shared((BN, K), "bfloat16", scope="shared.l1")
c = T.alloc_fragment((BM, BN), "float32", scope="shared.l0c")
T.annotate_layout({c: make_ascend_l0c_layout(c)})
```

`scope="shared.l1"` 将输入块放在 L1，`scope="shared.l0c"` 将累加结果放在 L0C；`make_ascend_l0c_layout` 为结果配置 L0C 所需的存储布局。`T.alloc_l0c` 默认已完成这一布局配置，所以原写法不需要额外标注。

**后续代码统一使用 `T.alloc_l1` 和 `T.alloc_l0c` 来组织，直接体现输入位于 L1、累加结果位于 L0C。** 理解通用写法有助于区分 API 名称、内存位置与存储布局。

## 2. T.gemm 参数说明

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">参数</th>
<th style="text-align: left; vertical-align: top;">含义</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">a、b</td>
<td style="text-align: left; vertical-align: top;">L1 中的两个 bf16 输入块</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">c</td>
<td style="text-align: left; vertical-align: top;">L0C 中的 fp32 结果或部分和</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">transpose_A=False（默认）</td>
<td style="text-align: left; vertical-align: top;">左输入保持原方向</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">transpose_B=True</td>
<td style="text-align: left; vertical-align: top;">计算中使用右输入的转置；B 实际仍按 <code>(N,K)</code> 存储</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">clear_accum=True</td>
<td style="text-align: left; vertical-align: top;">开始一个新输出块的计算</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">clear_accum=(kt==0)</td>
<td style="text-align: left; vertical-align: top;">K 首段重新开始，后续段累加到已有结果</td>
</tr>
</tbody>
</table>

T.gemm 由后端完成操作数准备和 Cube 计算，其中 `clear_accum` 是控制 K 方向累加的参数。

## 3. tiling 分核与 K 切分

### 先实现完整 K 的基础版本

取 `M=512、N=36864、K=256`，输出 C 的形状为 `(512,36864)`。选择 `BM=BN=128`，先按输出的行、列划分为 `4×288=1152` 个 tile，再分配给各核。每核依次完成 144 个输出块。

输出块的 fp32 结果占 `4×BM×BN=64 KiB` L0C；两个 bf16 输入块占 `2×(BM+BN)×K=128 KiB` L1，完整 K 可以直接读入。M/N 切分控制输出块大小，K 则决定每次需要准备多少输入。这一组参数便于理解计算过程，实际选取还要结合复用和流水配置。

### 从任务编号定位输出块

`mi` 是输出 tile 的行块编号（0～3），`ni` 是列块编号（0～287），均从 0 开始。该 tile 对应 `C[mi×128:(mi+1)×128, ni×128:(ni+1)×128]`。把输出 tile 按行优先编号 `idx=mi×288+ni`。

core bx 的第 it 轮处理 `idx=bx×144+it`，再由 `mi=idx//288`、`ni=idx%288` 还原二维位置。例如 core1 首轮 idx=144，对应 `C[0:128,18432:18560]`。

![输出分块与每核任务](images/04.02_base_tasks.svg)

下面给出基础实现。每轮计算不同的输出块，一次覆盖完整 K，因此每次 `clear_accum=True`，随后写回该块结果。

```python
import tilelang
import tilelang.language as T


def basic_gemm(M=512, N=36864, K=256, BM=128, BN=128):
    tasks = (M // BM) * (N // BN)

    @T.prim_func
    def main(
        A: T.Tensor((M, K), 'bfloat16'),
        B: T.Tensor((N, K), 'bfloat16'),
        C: T.Tensor((M, N), 'float32'),
    ):
        with T.Kernel(8) as bx:
            a = T.alloc_l1((BM, K), 'bfloat16')
            b = T.alloc_l1((BN, K), 'bfloat16')
            c = T.alloc_l0c((BM, BN), 'float32')

            for it in T.Pipelined(tasks // 8, num_stages=1):
                idx = bx * (tasks // 8) + it
                mi = idx // (N // BN)
                ni = idx % (N // BN)
                T.copy(A[mi * BM : (mi + 1) * BM, :], a)
                T.copy(B[ni * BN : (ni + 1) * BN, :], b)
                T.gemm(a, b, c, transpose_B=True, clear_accum=True)

                T.copy(c, C[mi * BM : (mi + 1) * BM, ni * BN : (ni + 1) * BN])

    return main
```

[可执行工厂与 Host 校验](src/gemm_kernel.py)。`num_stages=1` 保留一份内存，4.3 将只调整输出块循环的 buffer 数量。

### 验证基础实现

使用相同量化后的输入计算 fp32 参考 AB<sup>T</sup>；随机、全零、常数和带符号周期输入各连续调用两次，对全部输出以 `rtol=atol=1e-4` 比较。

In [1]:
import subprocess, sys
p = subprocess.run([sys.executable, 'src/run_gemm.py', 'basic'], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print("\n".join(line for line in p.stdout.splitlines()
                       if not line.startswith("Loading tilelang libs from dev root:")))
p.check_returncode()

[W917 20:44:11.343887964 FunctionLoader.cpp:48] Warning: LD_PRELOAD detected, FunctionLoader prefers RTLD_DEFAULT for symbol resolution. (function operator())
512_36864_256_128_128_s1_p0 random: full-output PASS; shape=(512, 36864); max_abs_error=2.3841858e-06
512_36864_256_128_128_s1_p0 zero: full-output PASS; shape=(512, 36864); max_abs_error=0
512_36864_256_128_128_s1_p0 constant: full-output PASS; shape=(512, 36864); max_abs_error=0
512_36864_256_128_128_s1_p0 pattern: full-output PASS; shape=(512, 36864); max_abs_error=0
Verification passed!



### 如果 K 增大，完整输入块放不下怎么办？

保持 M、N 和输出 tile 不变，将 K 从 256 增加到 2048。每个输出块对应的两个输入块现在需要：

`2×(128+128)×2048 = 1,048,576 byte = 1024 KiB`。

本例 L1 容量预算为 512 KiB，无法同时存放这两块完整输入。这里约束的是 Cube 输入所在的 L1。输出的形状没有改变，c 仍占 64 KiB L0C。

办法是沿 K 切成 `BK=128` 的小段，每轮只占 64 KiB L1，分 16 轮读入和计算。对行块 mi、列块 ni，令 $A_{mi,kt}$ 表示 A 的 `[mi×BM:(mi+1)×BM, kt×BK:(kt+1)×BK]` 区域，$B_{ni,kt}$ 表示 B 的 `[ni×BN:(ni+1)×BN, kt×BK:(kt+1)×BK]` 区域。第 kt 段对当前输出块的贡献为：

$$
P_{kt}=\left[A_{mi,kt}\right]\cdot\left[B_{ni,kt}\right]^{T},\qquad C_{mi,ni}=\sum_{kt=0}^{K/BK-1}P_{kt}.
$$

每个 $P_{kt}$ 的形状都是 `(BM,BN)`，只累加到当前输出块。

![K 切分与同一输出块累加](images/04.02_k_accumulate.svg)

第一轮令 `c=P_0`，后续依次累加 `P_1…P_15`，所以 `clear_accum=(kt==0)`。

**只有全部 16 段计算完成后才写回 c。**

若每轮清累加，会只保留最后一段；若不在首轮清累加，便没有可靠的初始结果。

### K 切分的完整实现

外层仍遍历本核的输出块，新增的内层循环只负责当前输出块的 K 方向累加。它与外层“每轮得到一个新输出块”的作用不同。

```python
import tilelang
import tilelang.language as T


def k_demo_gemm(M=512, N=36864, K=2048, BM=128, BN=128, BK=128):
    if min(M, N, K, BM, BN, BK) <= 0 or M % BM or N % BN or K % BK:
        raise ValueError('Positive dimensions and complete tiles are required')
    tasks = (M // BM) * (N // BN)
    if tasks % 8 or 2 * (BM + BN) * BK > 512 * 1024 or 4 * BM * BN > 256 * 1024:
        raise ValueError('Check task division and L1/L0C capacity')

    @T.prim_func
    def main(
        A: T.Tensor((M, K), 'bfloat16'),
        B: T.Tensor((N, K), 'bfloat16'),
        C: T.Tensor((M, N), 'float32'),
    ):
        with T.Kernel(8) as bx:
            a = T.alloc_l1((BM, BK), 'bfloat16')
            b = T.alloc_l1((BN, BK), 'bfloat16')
            c = T.alloc_l0c((BM, BN), 'float32')

            for it in T.Pipelined(tasks // 8, num_stages=1):
                idx = bx * (tasks // 8) + it
                mi = idx // (N // BN)
                ni = idx % (N // BN)
                for kt in T.Pipelined(K // BK, num_stages=1):
                    T.copy(A[mi * BM : (mi + 1) * BM, kt * BK : (kt + 1) * BK], a)
                    T.copy(B[ni * BN : (ni + 1) * BN, kt * BK : (kt + 1) * BK], b)
                    T.gemm(a, b, c, transpose_B=True, clear_accum=(kt == 0))

                T.copy(c, C[mi * BM : (mi + 1) * BM, ni * BN : (ni + 1) * BN])

    return main
```

### 验证分段累加

验证用例为 `(512,36864,2048)`，保持相同 BM、BN，并设置 BK=128。四类输入和精度阈值与基础版相同，连续调用两次还可检查不同调用之间是否有累加残留。<a href="./src/run_gemm.py">完整验证入口</a>。

```bash
python src/run_gemm.py k_demo
```

In [2]:
import subprocess, sys
p = subprocess.run([sys.executable, 'src/run_gemm.py', 'k_demo'], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print("\n".join(line for line in p.stdout.splitlines()
                       if not line.startswith("Loading tilelang libs from dev root:")))
p.check_returncode()

[W917 20:44:37.704356405 FunctionLoader.cpp:48] Warning: LD_PRELOAD detected, FunctionLoader prefers RTLD_DEFAULT for symbol resolution. (function operator())
2026-09-17 20:44:43  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:131): TileLang begins to compile kernel `main` with `out_idx=[2]`
2026-09-17 20:44:55  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:139): TileLang completes to compile kernel `main`
K split (512, 36864, 2048) random: full-output PASS; shape=(512, 36864); max_abs_error=9.5367432e-06
K split (512, 36864, 2048) zero: full-output PASS; shape=(512, 36864); max_abs_error=0
K split (512, 36864, 2048) constant: full-output PASS; shape=(512, 36864); max_abs_error=0
K split (512, 36864, 2048) pattern: full-output PASS; shape=(512, 36864); max_abs_error=0
Verification passed!



## 小结

本节先完成完整 K 的 GEMM，并验证 `(512,36864,256)` 的全部输出；再把 K 改为 2048，从 L1 输入容量不足引出 K 切分和部分积累加。下一节回到完整 K 的基础用例，研究不同输出块之间的搬运与计算如何形成流水。

## 课后练习

1. 基础用例中，core2 第 3 轮（it=2）的 idx、mi、ni 和输出区域分别是什么？
2. 保持 BM=BN=128、K=2048，BK 改为 256 后，输入 L1 占用、K 轮数和 L0C 占用各是多少？
3. 为什么每轮清累加的错误可能在 K 只有一段时未被发现？

In [3]:
from pathlib import Path
print(Path("answer/04.02_answer.txt").read_text())

1. core2、it=2：idx=2×144+2=290，mi=1，ni=2，输出 C[128:256,256:384]。
2. BK=256 时，输入 L1=2×(128+128)×256=128 KiB；K=2048 需要8轮；L0C仍为64 KiB。
3. K只有一段时，首轮也是末轮；每轮清累加不会丢失更早的部分积，多段时才会暴露问题。

